# dummygrad on Colab

Builds dummygrad with CUDA, runs the test suite and the quickstart, trains a small transformer, and shows how to use dummygrad from your own project.

**Runtime → Change runtime type → T4 GPU** (any NVIDIA GPU works), then **Runtime → Run all**.


In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
# Branch or tag to test: edit the text box on the right, then run
BRANCH = "perf/gpu-kernels"  # @param {type:"string"}

%cd /content
!rm -rf dummygrad && git clone -q --depth 1 -b {BRANCH} https://github.com/Keplercodes01/dummygrad.git
%cd /content/dummygrad
!git log --oneline -1

In [ ]:
# Configure + build (CUDA architecture is detected from the attached GPU)
!apt-get -qq install -y libopenblas-dev > /dev/null
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | grep -E "CUDA|BLAS|Error"
!cmake --build build -j$(nproc) 2>&1 | grep -E "error|Error|Built target"

## Test suite

CPU gradient checks against finite differences, CUDA vs CPU parity, autograd engine tests, and end-to-end training. Ends with `N passed, 0 failed`.

In [ ]:
!./build/test_engine

## Benchmark vs PyTorch

Full training steps (forward, backward, AdamW) of the same ModernTransformer architecture in dummygrad and in eager PyTorch. `copies/step` and `MB/step` count host-device transfers in dummygrad: each one is an op falling back to the CPU.

In [ ]:
!./build/dg_bench

In [ ]:
!python bench/bench_torch.py

## Quickstart

A tour of the API: autograd on a tiny expression, then a two-layer network learning XOR. Source: [`examples/quickstart.cpp`](https://github.com/Keplercodes01/dummygrad/blob/main/examples/quickstart.cpp).

In [ ]:
!./build/quickstart

## Training template

A complete run: a character-level language model with validation, AdamW with warmup and cosine decay, gradient clipping, metric logging, checkpoints and sampling. Source: [`examples/train_lm.cpp`](https://github.com/Keplercodes01/dummygrad/blob/main/examples/train_lm.cpp). Point it at any text file with `!cd build && ./train_lm my_text.txt`.

In [ ]:
!cd build && ./train_lm

## Training demo

Trains a 2-layer GQA transformer on a synthetic next-token task, saves a SafeTensors checkpoint, and reloads it. The loss should fall steadily.

In [ ]:
!cd build && ./train_colab

## Use dummygrad in your own project

Creates a separate CMake project that depends on dummygrad with `FetchContent`, exactly as in the README. `FETCHCONTENT_SOURCE_DIR_DUMMYGRAD` points it at the checkout above so it tests the same code; drop that flag in your own project to download the tagged release. (Builds dummygrad again, ~3 minutes.)

In [ ]:
import os
os.makedirs('/content/my_app', exist_ok=True)
open('/content/my_app/CMakeLists.txt', 'w').write("cmake_minimum_required(VERSION 3.18)\nproject(my_app LANGUAGES CXX)\nset(CMAKE_CXX_STANDARD 17)\n\ninclude(FetchContent)\nFetchContent_Declare(dummygrad\n    GIT_REPOSITORY https://github.com/Keplercodes01/dummygrad.git\n    GIT_TAG        v0.4.0)\nFetchContent_MakeAvailable(dummygrad)\n\nadd_executable(my_app main.cpp)\ntarget_link_libraries(my_app PRIVATE dummygrad::dummygrad)\n")
open('/content/my_app/main.cpp', 'w').write("#include \"dummy_core.h\"\n#include <iostream>\n\nint main() {\n    auto x = randn({2, 3});\n    auto w = randn({3, 4});\n    auto loss = sum(sum(matmul(x, w), -1), -2);\n    loss->backward();\n    std::cout << \"my_app: dummygrad works, dloss/dw has \" << w->grad->size() << \" entries\\n\";\n    return 0;\n}\n")

!rm -rf /content/my_app/build
!cmake -S /content/my_app -B /content/my_app/build -DCMAKE_BUILD_TYPE=Release -DFETCHCONTENT_SOURCE_DIR_DUMMYGRAD=/content/dummygrad 2>&1 | grep -E -A6 "CUDA found|Error"
!cmake --build /content/my_app/build -j$(nproc) 2>&1 | grep -E "error|Error|Built target"
!/content/my_app/build/my_app